# Data Profiling

Melakukan profiling pada data Flood Prediction South Sulawesi menggunakan Polars dan DuckDB.

In [32]:
import os
import polars as pl
import duckdb

DATA_PATH = '../data/modis_flood_features.csv'


## Profiling menggunakan Polars

In [33]:
# Load data lazily
df_lazy = pl.scan_csv(DATA_PATH, ignore_errors=True)

# Tampilkan skema
print(df_lazy.collect_schema())

Schema({'date': String, 'lon': Float64, 'lat': Float64, 'flooded': Float64, 'jrc_perm_water': Float64, 'precip_1d': Float64, 'precip_3d': Float64, 'NDVI': Float64, 'NDWI': Float64, 'landcover': Float64, 'elevation': Float64, 'slope': Float64, 'aspect': Float64, 'upstream_area': Float64, 'TWI': Float64, 'target': Int64})


In [34]:
# Tampilkan beberapa baris pertama
df_lazy.head().collect()

date,lon,lat,flooded,jrc_perm_water,precip_1d,precip_3d,NDVI,NDWI,landcover,elevation,slope,aspect,upstream_area,TWI,target
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""2003-12-10""",120.478677,-6.495942,1.0,1.0,9.645125,28.935376,6190.93062,-0.051446,17.0,0.0,0.116685,270.0,0.03199,1.042098,0
"""2003-12-10""",120.480923,-6.495942,1.0,1.0,9.645125,28.935376,6190.93062,-0.051446,17.0,1.0,0.839713,236.480074,0.03199,1.042098,0
"""2003-12-10""",120.478677,-6.493697,0.0,1.0,9.645125,28.935376,6190.93062,0.028399,17.0,0.0,0.46673,270.0,0.03199,1.042098,0
"""2003-12-10""",120.480923,-6.493697,1.0,0.0,9.645125,28.935376,6190.93062,0.028399,17.0,4.0,0.939179,240.413746,0.021235,0.258796,1
"""2003-12-10""",120.478677,-6.491451,0.0,1.0,9.645125,28.935376,6190.93062,-0.002506,17.0,0.0,0.583403,270.0,0.03199,1.042098,0


In [35]:
# Deskripsi data (statistik)
df_lazy.collect().describe()

statistic,date,lon,lat,flooded,jrc_perm_water,precip_1d,precip_3d,NDVI,NDWI,landcover,elevation,slope,aspect,upstream_area,TWI,target
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""1025801""",1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,120.177808,-3.961987,0.112058,0.093222,11.898757,42.717388,5995.309842,0.24714,11.128318,125.314918,1.062715,155.005117,16.332991,2.843527,0.045189
"""std""",null,0.52951,1.018204,0.315438,0.290743,14.388524,31.635749,2020.938202,0.116986,3.762334,274.432429,1.597632,106.337275,149.262247,2.500571,0.207719
"""min""","""2003-12-10""",119.353537,-6.495942,0.0,0.0,0.0,0.0,-2000.0,-0.385959,1.0,0.0,0.0,0.0,0.008497,0.000023,0.0
"""25%""",null,119.791466,-4.766685,0.0,0.0,0.0,20.839432,4912.0,0.183155,9.0,11.0,0.259303,63.489535,0.023397,0.933785,0.0
"""50%""",null,120.153038,-4.000872,0.0,0.0,7.932389,39.32737,6417.0,0.255126,12.0,25.0,0.492959,153.378298,0.08753,2.119129,0.0
"""75%""",null,120.357405,-2.853274,0.0,0.0,20.105299,60.266768,7552.5,0.326531,14.0,90.0,1.187444,243.542309,0.58918,4.174976,0.0
"""max""","""2015-02-05""",121.790218,-1.894322,1.0,1.0,107.288399,220.21907,10000.0,0.598126,17.0,2694.0,26.720255,359.348113,5128.527832,14.996599,1.0


## Profiling menggunakan DuckDB `SUMMARIZE`

In [36]:
# Menjalankan perintah SUMMARIZE pada file CSV dengan DuckDB
query = f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}')"
duckdb.query(query).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,date,DATE,2003-12-10,2015-02-05,7,2008-04-07 17:08:34.285714,None,2004-05-07,2007-06-16,2013-01-03,1025801,0.0
1,lon,DOUBLE,119.353537330435,121.790217538609,978,120.17780767018337,0.5295104965429103,119.79282354256978,120.15265565170873,120.35692983308266,1025801,0.0
2,lat,DOUBLE,-6.49594239828929,-1.89432235538704,2049,-3.9619865334557076,1.0182037099318137,-4.771869029582089,-4.001037695362841,-2.85192288194953,1025801,0.0
3,flooded,DOUBLE,0.0,1.0,2,0.11205779678514644,0.3154377021902998,0.0,0.0,0.0,1025801,0.0
4,jrc_perm_water,DOUBLE,0.0,1.0,2,0.0932217847321264,0.2907431271567862,0.0,0.0,0.0,1025801,0.0
5,precip_1d,DOUBLE,0.0,107.288398742676,5669,11.898757013442728,14.388523849003391,0.0,7.893404223799392,20.10910395611454,1025801,0.0
6,precip_3d,DOUBLE,0.0,220.21907043457,9405,42.717387874977604,31.635748575183488,20.876724668476697,39.36308277930042,60.311907518136856,1025801,0.0
7,NDVI,DOUBLE,-2000.0,10000.0,25932,5995.309841795773,2020.9382019573575,4914.362273167685,6418.610499491676,7550.880334743448,1025801,0.0
8,NDWI,DOUBLE,-0.385959088802338,0.598126113414764,585997,0.24713965297360463,0.11698595463245536,0.18298698061021332,0.2550702551610587,0.326453433240434,1025801,0.0
9,landcover,DOUBLE,1.0,17.0,15,11.12831826055931,3.7623340536655445,9.0,12.0,14.0,1025801,0.0
